# Vérification des données prétraitées — « pluie prévue »

Ce carnet lit les sorties de `pretraiter.py` (dossier
`donnees_pluie_prevue/preprocessed/`) : cubes NetCDF CHIRPS / IMERG, indices
ENSO, et tables de features au format CSV.

Objectif : vérifier que les données sont exploitables avant de modéliser.
Région : Afrique de l'Ouest / Sahel, lon [-20, 20], lat [-10, 25].


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from netCDF4 import Dataset
from pathlib import Path

OUT = Path("donnees_pluie_prevue/preprocessed")
FIG = OUT / "figures"
FIG.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.3})

cube = Dataset(OUT / "chirps_sahel_mensuel.nc")
pluie = np.ma.filled(cube["precipitation"][:], np.nan)   # (time, lat, lon) mm/mois
lat = np.asarray(cube["lat"][:])
lon = np.asarray(cube["lon"][:])
time = pd.to_datetime(cube["time"][:], unit="D", origin="1970-01-01")

print("cube CHIRPS :", pluie.shape, "|", time[0].date(), "->", time[-1].date())
print("grille      :", lat.size, "lat x", lon.size, "lon  (",
      round(float(lat[1] - lat[0]), 3), "°)")
print("valeurs     :", f"{np.isfinite(pluie).mean():.1%} valides",
      "| max", f"{np.nanmax(pluie):.0f} mm/mois")


## 1. La table de features

Une ligne = un mois × une maille de 0,5°. Les mailles océaniques ont été
retirées (CHIRPS ne couvre que les terres).


In [ ]:
df = pd.read_csv(OUT / "features_chirps_sahel.csv", parse_dates=["date"])

print(f"{len(df):,} lignes | {df.date.nunique()} mois | "
      f"{df.groupby(['lat', 'lon']).ngroups:,} mailles")
print(f"période : {df.date.min():%Y-%m} -> {df.date.max():%Y-%m}")
df.head()


In [ ]:
# part de valeurs manquantes par colonne (les décalages créent des trous aux bords)
manquant = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print(manquant[manquant > 0].to_string())


## 2. Cycle saisonnier de la pluie (moyenne sur la région)

In [ ]:
saison = df.groupby("month")["rain_mm"].mean()

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(saison.index, saison.values, color="#3b7ea1")
ax.set(xlabel="mois", ylabel="pluie moyenne (mm/mois)",
       title="Cycle saisonnier moyen — bbox Sahel 1981-1985",
       xticks=range(1, 13))
ax.axvspan(5.5, 9.5, color="orange", alpha=0.12, lw=0)
ax.text(7.5, saison.max() * 0.95, "mousson", ha="center", color="#b06a00")
fig.tight_layout()
fig.savefig(FIG / "cycle_saisonnier.png")


## 3. Carte de la pluie annuelle moyenne

In [ ]:
annees = slice(0, 48)                      # 1981-1984 (4 années complètes)
annuel = np.nansum(pluie[annees], axis=0) / 4

fig, ax = plt.subplots(figsize=(7, 5))
m = ax.pcolormesh(lon, lat, annuel, cmap="YlGnBu", vmin=0, vmax=2500, shading="auto")
fig.colorbar(m, ax=ax, label="mm/an", shrink=0.8)
ax.set(xlabel="longitude", ylabel="latitude", title="Pluie annuelle moyenne 1981-1984")
ax.grid(False)
fig.tight_layout()
fig.savefig(FIG / "pluie_annuelle.png")


### Contrôle indépendant par stations de référence

Comparaison avec des climatologies connues. Attention : 1981-1984 correspond à
une période de **sécheresse au Sahel**, donc des valeurs 20 à 35 % sous les
normales longues sont attendues à Dakar, Tombouctou ou Niamey.

In [ ]:
refs = {
    "Dakar":       (14.7, -17.4,  400),
    "Tombouctou":  (16.8,  -3.0,  200),
    "Niamey":      (13.5,   2.1,  550),
    "Ouagadougou": (12.4,  -1.5,  800),
    "Conakry":     ( 9.5, -13.7, 3800),
}
lignes = []
for nom, (la, lo, ref) in refs.items():
    i = int(np.argmin(abs(lat - la)))
    j = int(np.argmin(abs(lon - lo)))
    val = annuel[i, j]
    lignes.append({"station": nom, "calculé (mm/an)": round(val),
                   "référence (mm/an)": ref, "écart (%)": round(100 * (val - ref) / ref)})
pd.DataFrame(lignes)


## 4. Indices ENSO

Les indices sont recalculés/vérifiés dans `indices_enso_mensuel.nc`.
Le contrôle croisé Niño 3.4 recalculé depuis ERSST v6 vs ONI officiel donne
r ≈ 0,99 : la chaîne SST est correcte.


In [ ]:
ind = Dataset(OUT / "indices_enso_mensuel.nc")
t_ind = pd.to_datetime(ind["time"][:], unit="D", origin="1970-01-01")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t_ind, ind["oni"][:], label="ONI (CPC)", lw=1.4)
ax.plot(t_ind, ind["nino34_anom"][:], label="Niño 3.4 (ERSST v6 recalculé)",
        lw=1, ls="--")
ax.axhline(0, color="k", lw=0.6)
for y0, y1, txt in [(1982.4, 1983.6, "El Niño 1982-83"), (1997.4, 1998.6, "El Niño 1997-98")]:
    ax.axvspan(pd.Timestamp(f"{y0:.0f}-06-01"), pd.Timestamp(f"{y1:.0f}-06-01"),
               color="red", alpha=0.08, lw=0)
ax.set(ylabel="anomalie SST (°C)", title="Indices ENSO")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "indices_enso.png")


## 5. Ce qu'on ne peut pas encore faire

- Corréler pluie et ENSO : seulement **4 saisons des pluies** complètes
  (1981-1984) — largement insuffisant.
- Fusionner CHIRPS et IMERG : les deux produits ne se recouvrent pas
  (1981-1985 vs 1998-1999).
- Calculer une climatologie robuste : 4 années seulement.

Il faut donc compléter les téléchargements (`resume_all.sh`, après avoir
corrigé le chemin vers `00_docs/01_chirps_v3_mensuel/`) puis relancer
`pretraiter.py` : le script reprend automatiquement tous les fichiers présents.
